# 🧬 Dark Manifold V23 — Real Biochemistry + Complete Training

## What V23 Fixes

**V22 was incomplete:**
- ❌ No optimizer
- ❌ No data generation  
- ❌ No training loop
- ❌ Random stoichiometry initialization

**V23 adds:**
- ✅ Real SBML data from Luthey-Schulten Lab (iMB155 model)
- ✅ Real stoichiometry matrix (304 metabolites × 244 reactions)
- ✅ Real gene-reaction associations from GPR
- ✅ Complete training loop with optimizer
- ✅ Proper loss function and evaluation

**Data source:** [Luthey-Schulten Lab](https://github.com/Luthey-Schulten-Lab/Minimal_Cell)  
**Paper:** "Fundamental behaviors emerge from simulations of a living minimal cell" (Cell, 2022)

In [ ]:
#@title 1️⃣ Setup & Download Real Data
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Install dependencies
!pip install python-libsbml openpyxl pandas torchdiffeq -q

# Clone Luthey-Schulten data
if not os.path.exists('Minimal_Cell'):
    print("📥 Downloading Luthey-Schulten Lab real data...")
    !git clone --depth 1 https://github.com/Luthey-Schulten-Lab/Minimal_Cell.git
    print("✅ Data downloaded!")
else:
    print("✅ Minimal_Cell data already present")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🖥️ Device: {device}")
print(f"🔥 PyTorch: {torch.__version__}")

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
#@title 2️⃣ Parse REAL SBML Model — Extract Stoichiometry
import libsbml
import pandas as pd
import re

class RealBiochemistry:
    """
    Parse iMB155 SBML model to extract REAL biochemistry:
    - Stoichiometry matrix S (metabolites × reactions)
    - Gene-Protein-Reaction (GPR) associations
    - Metabolite/reaction metadata
    """
    
    def __init__(self, sbml_path):
        self.doc = libsbml.readSBML(sbml_path)
        self.model = self.doc.getModel()
        
        if self.model is None:
            raise ValueError(f"Could not parse SBML: {sbml_path}")
        
        self._parse_model()
        print(f"📄 Loaded: {self.model.getId()}")
        print(f"   {self.n_metabolites} metabolites, {self.n_reactions} reactions, {self.n_genes} genes")
    
    def _parse_model(self):
        # Get metabolites
        self.met_ids = []
        self.met_names = []
        for i in range(self.model.getNumSpecies()):
            sp = self.model.getSpecies(i)
            self.met_ids.append(sp.getId())
            self.met_names.append(sp.getName())
        
        self.met_to_idx = {m: i for i, m in enumerate(self.met_ids)}
        self.n_metabolites = len(self.met_ids)
        
        # Get reactions and build stoichiometry
        self.rxn_ids = []
        self.rxn_names = []
        self.gprs = []
        
        S = np.zeros((self.n_metabolites, self.model.getNumReactions()))
        
        for j in range(self.model.getNumReactions()):
            rxn = self.model.getReaction(j)
            self.rxn_ids.append(rxn.getId())
            self.rxn_names.append(rxn.getName())
            
            # GPR
            gpr = ''
            fbc = rxn.getPlugin('fbc')
            if fbc:
                gpa = fbc.getGeneProductAssociation()
                if gpa and gpa.getAssociation():
                    gpr = gpa.getAssociation().toInfix()
            self.gprs.append(gpr)
            
            # Stoichiometry
            for k in range(rxn.getNumReactants()):
                ref = rxn.getReactant(k)
                sp_id = ref.getSpecies()
                if sp_id in self.met_to_idx:
                    S[self.met_to_idx[sp_id], j] = -ref.getStoichiometry()
            
            for k in range(rxn.getNumProducts()):
                ref = rxn.getProduct(k)
                sp_id = ref.getSpecies()
                if sp_id in self.met_to_idx:
                    S[self.met_to_idx[sp_id], j] = ref.getStoichiometry()
        
        self.S = S
        self.n_reactions = len(self.rxn_ids)
        
        # Parse genes from GPR
        all_genes = set()
        for gpr in self.gprs:
            genes = re.findall(r'(?:G_)?JCVISYN3A_(\d+)', gpr)
            all_genes.update(genes)
        
        self.gene_ids = sorted(list(all_genes))
        self.gene_to_idx = {g: i for i, g in enumerate(self.gene_ids)}
        self.n_genes = len(self.gene_ids)
        
        # Build gene-reaction matrix
        G = np.zeros((self.n_genes, self.n_reactions))
        for j, gpr in enumerate(self.gprs):
            genes = re.findall(r'(?:G_)?JCVISYN3A_(\d+)', gpr)
            for g in genes:
                if g in self.gene_to_idx:
                    G[self.gene_to_idx[g], j] = 1.0
        self.G = G
    
    def get_key_metabolites(self):
        """Return indices of important metabolites for monitoring."""
        key_names = ['atp', 'adp', 'amp', 'nad', 'nadh', 'pyr', 'lac', 'g6p', 'f6p']
        indices = []
        names = []
        for i, mid in enumerate(self.met_ids):
            for key in key_names:
                if key in mid.lower() and '_c' in mid:  # Cytoplasmic
                    indices.append(i)
                    names.append(mid)
                    break
        return indices, names

# Load real data
sbml_path = 'Minimal_Cell/CME_ODE/model_data/iMB155_NoH2O.xml'
biochem = RealBiochemistry(sbml_path)

print(f"\n📊 Stoichiometry Matrix S: {biochem.S.shape}")
print(f"   Non-zeros: {np.count_nonzero(biochem.S)} ({100*np.count_nonzero(biochem.S)/biochem.S.size:.1f}% sparse)")
print(f"\n📊 Gene-Reaction Matrix G: {biochem.G.shape}")
print(f"   Genes with annotations: {len(biochem.gene_ids)}")

In [ ]:
#@title 3️⃣ Configuration (REAL dimensions from SBML)

# REAL dimensions from parsed data
N_METABOLITES = min(biochem.n_metabolites, 150)  # Use top 150 for tractability
N_REACTIONS = min(biochem.n_reactions, 150)
N_GENES = min(biochem.n_genes, 100)
PERTURBATION_DIM = 12

# Perception (from V22)
D_MODEL = 256
N_LATENTS = 64
SSM_DIM = 128
N_EXPERTS = 8
TOP_K = 2

# Dynamics (from V22)
KOOPMAN_DIM = 128
FLOW_DIM = 64

# Reasoning (from V22)
MAX_RULES = 100
N_HOPS = 3
HIERARCHY_LEVELS = 4

# Training (ADDED - was missing in V22!)
N_EPOCHS = 2000
LEARNING_RATE = 2e-4
BATCH_SIZE = 8
SEQ_LEN = 20

print("📋 V23 Configuration:")
print(f"   Metabolites: {N_METABOLITES} (from {biochem.n_metabolites} in SBML)")
print(f"   Reactions: {N_REACTIONS} (from {biochem.n_reactions} in SBML)")
print(f"   Genes: {N_GENES} (from {biochem.n_genes} in SBML)")
print(f"   Training epochs: {N_EPOCHS}")

In [ ]:
#@title 4️⃣ V22 Mechanisms — State Space Model (Mamba-style)

class SelectiveSSM(nn.Module):
    """Mamba-style selective state space model. O(n) not O(n²)."""
    
    def __init__(self, d_model, d_state, d_conv=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        
        self.in_proj = nn.Linear(d_model, d_model * 2)
        self.conv1d = nn.Conv1d(d_model, d_model, d_conv, padding=d_conv-1, groups=d_model)
        self.x_proj = nn.Linear(d_model, d_state * 2 + 1)
        self.A_log = nn.Parameter(torch.randn(d_model, d_state))
        self.D = nn.Parameter(torch.ones(d_model))
        self.out_proj = nn.Linear(d_model, d_model)
    
    def forward(self, x):
        batch, seq_len, d = x.shape
        
        xz = self.in_proj(x)
        x_branch, z = xz.chunk(2, dim=-1)
        
        x_conv = self.conv1d(x_branch.transpose(1, 2))[:, :, :seq_len].transpose(1, 2)
        x_conv = F.silu(x_conv)
        
        ssm_params = self.x_proj(x_conv)
        B = ssm_params[..., :self.d_state]
        C = ssm_params[..., self.d_state:2*self.d_state]
        dt = F.softplus(ssm_params[..., -1:])
        
        A = -torch.exp(self.A_log)
        
        # Simplified scan
        h = torch.zeros(batch, d, self.d_state, device=x.device)
        outputs = []
        
        for t in range(seq_len):
            dt_t = dt[:, t:t+1, :].expand(-1, d, -1)
            B_t = B[:, t, :].unsqueeze(1).expand(-1, d, -1)
            C_t = C[:, t, :].unsqueeze(1).expand(-1, d, -1)
            x_t = x_conv[:, t, :].unsqueeze(-1)
            
            h = h * torch.exp(A * dt_t) + x_t * B_t * dt_t
            y_t = (h * C_t).sum(dim=-1)
            outputs.append(y_t)
        
        y = torch.stack(outputs, dim=1)
        y = y + x_conv * self.D
        
        output = y * F.silu(z)
        return self.out_proj(output)

print("✅ SelectiveSSM defined")

In [ ]:
#@title 5️⃣ V22 Mechanisms — Perceiver IO

class PerceiverIO(nn.Module):
    """Perceiver IO with fixed latent bottleneck."""
    
    def __init__(self, input_dim, n_latents, d_model, n_heads=8, n_layers=4):
        super().__init__()
        self.n_latents = n_latents
        self.d_model = d_model
        
        self.input_proj = nn.Linear(input_dim, d_model)
        self.latents = nn.Parameter(torch.randn(n_latents, d_model) * 0.02)
        
        # Cross-attention: latents attend to inputs
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        
        # Self-attention layers
        self.self_attn_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model, n_heads, d_model*4, batch_first=True)
            for _ in range(n_layers)
        ])
    
    def forward(self, x):
        batch_size = x.shape[0]
        
        inputs = self.input_proj(x)
        latents = self.latents.unsqueeze(0).expand(batch_size, -1, -1)
        
        # Cross-attention
        latents, _ = self.cross_attn(latents, inputs, inputs)
        
        # Self-attention
        for layer in self.self_attn_layers:
            latents = layer(latents)
        
        return latents

print("✅ PerceiverIO defined")

In [ ]:
#@title 6️⃣ V22 Mechanisms — Sparse MoE

class SparseMoE(nn.Module):
    """Sparse Mixture of Experts with top-k routing."""
    
    def __init__(self, d_model, n_experts, top_k, hidden_dim):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        
        self.router = nn.Linear(d_model, n_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, d_model)
            ) for _ in range(n_experts)
        ])
    
    def forward(self, x):
        batch, seq_len, d = x.shape
        x_flat = x.view(-1, d)
        
        logits = self.router(x_flat)
        weights, indices = torch.topk(F.softmax(logits, dim=-1), self.top_k)
        weights = weights / weights.sum(dim=-1, keepdim=True)
        
        # Auxiliary load balancing loss
        aux_loss = self._load_balance_loss(logits)
        
        output = torch.zeros_like(x_flat)
        for i in range(self.top_k):
            expert_idx = indices[:, i]
            expert_weight = weights[:, i:i+1]
            for j in range(self.n_experts):
                mask = (expert_idx == j)
                if mask.any():
                    output[mask] += expert_weight[mask] * self.experts[j](x_flat[mask])
        
        return output.view(batch, seq_len, d), aux_loss
    
    def _load_balance_loss(self, logits):
        probs = F.softmax(logits, dim=-1)
        avg_prob = probs.mean(dim=0)
        return (avg_prob * torch.log(avg_prob + 1e-10)).sum() * -1

print("✅ SparseMoE defined")

In [ ]:
#@title 7️⃣ V22 Mechanisms — Koopman Operator

class KoopmanOperator(nn.Module):
    """Linearize nonlinear dynamics in lifted space."""
    
    def __init__(self, state_dim, lifted_dim, d_model):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(state_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, lifted_dim)
        )
        self.K = nn.Linear(lifted_dim, lifted_dim, bias=False)  # Koopman matrix
        self.decoder = nn.Sequential(
            nn.Linear(lifted_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, state_dim)
        )
    
    def forward(self, x):
        z = self.encoder(x)  # Lift to Koopman space
        z_next = self.K(z)   # Linear evolution
        x_next = self.decoder(z_next)  # Project back
        return x_next

print("✅ KoopmanOperator defined")

In [ ]:
#@title 8️⃣ V22 Mechanisms — Neuro-Symbolic Reasoning

class NeuroSymbolic(nn.Module):
    """Combine neural networks with symbolic logic."""
    
    def __init__(self, n_predicates, n_rules, d_model):
        super().__init__()
        self.predicate_net = nn.Linear(d_model, n_predicates)
        self.rule_weights = nn.Parameter(torch.randn(n_rules, n_predicates * 2))
        self.rule_output = nn.Linear(n_rules, d_model)
    
    def forward(self, x):
        predicates = torch.sigmoid(self.predicate_net(x))
        pred_neg = 1 - predicates
        pred_all = torch.cat([predicates, pred_neg], dim=-1)
        
        rule_act = torch.sigmoid(self.rule_weights)
        rule_sat = (pred_all.unsqueeze(1) * rule_act.unsqueeze(0)).sum(dim=-1)
        
        output = self.rule_output(rule_sat)
        return output, predicates, rule_sat

print("✅ NeuroSymbolic defined")

In [ ]:
#@title 9️⃣ V22 Mechanisms — Program Synthesis

class ProgramSynthesis(nn.Module):
    """Learn interpretable metabolic rules."""
    
    def __init__(self, state_dim, max_rules, d_model):
        super().__init__()
        self.rule_bank = nn.Parameter(torch.randn(max_rules, state_dim) * 0.1)
        self.rule_threshold = nn.Parameter(torch.zeros(max_rules))
        self.rule_effect = nn.Parameter(torch.randn(max_rules, state_dim) * 0.1)
        self.rule_weight = nn.Linear(d_model, max_rules)
        self.context_embed = nn.Linear(state_dim, d_model)
    
    def forward(self, state):
        context = self.context_embed(state)
        weights = torch.softmax(self.rule_weight(context), dim=-1)
        
        conditions = (state.unsqueeze(1) * self.rule_bank.unsqueeze(0)).sum(dim=-1)
        satisfaction = torch.sigmoid(conditions - self.rule_threshold)
        
        weighted_sat = satisfaction * weights
        effect = (weighted_sat.unsqueeze(-1) * self.rule_effect.unsqueeze(0)).sum(dim=1)
        
        return effect, satisfaction, weights

print("✅ ProgramSynthesis defined")

In [ ]:
#@title 🔟 V22 Mechanisms — Hierarchical Planner

class HierarchicalPlanner(nn.Module):
    """Multi-level planning: Goals → Strategies → Pathways → Reactions."""
    
    def __init__(self, n_levels, dims, d_model):
        super().__init__()
        self.n_levels = n_levels
        self.dims = dims
        
        self.goal_setter = nn.Linear(d_model, dims[0])
        self.refiners = nn.ModuleList([
            nn.Linear(dims[i] + d_model, dims[i+1])
            for i in range(n_levels - 1)
        ])
        self.feedback = nn.ModuleList([
            nn.Linear(dims[i+1], dims[i])
            for i in range(n_levels - 1)
        ])
    
    def forward(self, state, context):
        plans = []
        goal = self.goal_setter(state)
        plans.append(goal)
        
        current = goal
        for refiner in self.refiners:
            refiner_input = torch.cat([current, context], dim=-1)
            refined = F.softmax(refiner(refiner_input), dim=-1)
            plans.append(refined)
            current = refined
        
        return plans[-1], plans

print("✅ HierarchicalPlanner defined")

In [ ]:
#@title 1️⃣1️⃣ Complete V23 Model — With REAL Stoichiometry

class DarkManifoldV23(nn.Module):
    """
    V23: V22 Architecture + Real Biochemistry + Complete Training
    
    Key innovation: Initialize stoichiometry from REAL SBML data,
    not random values!
    """
    
    def __init__(self, config, real_S=None, real_G=None):
        super().__init__()
        self.config = config
        
        n_met = config['n_metabolites']
        n_gene = config['n_genes']
        n_rxn = config.get('n_reactions', n_met)
        d = config['d_model']
        
        # === REAL BIOCHEMISTRY (NEW IN V23) ===
        if real_S is not None:
            # Use real stoichiometry from SBML
            S_subset = real_S[:n_met, :n_rxn]
            self.S = nn.Parameter(torch.tensor(S_subset, dtype=torch.float32), requires_grad=False)
            print(f"✅ Using REAL stoichiometry: {self.S.shape}")
        else:
            self.S = nn.Parameter(torch.randn(n_met, n_rxn) * 0.01)
            print("⚠️ Using random stoichiometry")
        
        if real_G is not None:
            G_subset = real_G[:n_gene, :n_rxn]
            self.G = nn.Parameter(torch.tensor(G_subset, dtype=torch.float32), requires_grad=False)
            print(f"✅ Using REAL gene-reaction: {self.G.shape}")
        else:
            self.G = nn.Parameter(torch.randn(n_gene, n_rxn) * 0.01)
        
        # Learnable flux prediction
        self.flux_predictor = nn.Sequential(
            nn.Linear(n_met + n_gene + config['perturbation_dim'], d),
            nn.GELU(),
            nn.Linear(d, n_rxn),
            nn.Tanh()  # Fluxes can be positive or negative
        )
        
        # === PERCEPTION (from V22) ===
        input_dim = n_met + n_gene + config['perturbation_dim']
        self.input_embed = nn.Linear(input_dim, d)
        
        self.ssm = SelectiveSSM(d, config['ssm_dim'])
        self.perceiver = PerceiverIO(1, config['n_latents'], d, n_heads=8, n_layers=4)
        self.moe = SparseMoE(d, config['n_experts'], config['top_k'], d * 4)
        
        # === DYNAMICS (from V22) ===
        self.koopman = KoopmanOperator(n_met + n_gene, config['koopman_dim'], d)
        
        # === REASONING (from V22) ===
        self.neurosymbolic = NeuroSymbolic(50, 30, d)
        self.program_synth = ProgramSynthesis(n_met + n_gene, config['max_rules'], d)
        self.planner = HierarchicalPlanner(
            config['hierarchy_levels'],
            [4, 8, 20, n_met],
            d
        )
        
        # === OUTPUT ===
        self.output_head = nn.Sequential(
            nn.Linear(d * 2, d),
            nn.GELU(),
            nn.Linear(d, n_met + n_gene)
        )
        
        self.uncertainty_head = nn.Sequential(
            nn.Linear(d, d // 2),
            nn.GELU(),
            nn.Linear(d // 2, n_met + n_gene),
            nn.Softplus()
        )
    
    def compute_stoichiometric_dynamics(self, metabolites, genes, perturbation):
        """
        Compute metabolite changes using REAL stoichiometry.
        dM/dt = S @ v  where v = flux vector
        """
        full_input = torch.cat([metabolites, genes, perturbation], dim=-1)
        v = self.flux_predictor(full_input)  # Predict fluxes
        
        # Apply stoichiometric constraints
        # Gene knockouts reduce flux through reactions they catalyze
        gene_effect = genes @ self.G  # (batch, n_rxn)
        gene_effect = torch.sigmoid(gene_effect)  # Scale to [0, 1]
        v = v * gene_effect  # Modulate fluxes by gene expression
        
        # dM/dt = S @ v
        dM = v @ self.S.T  # (batch, n_met)
        return dM, v
    
    def forward(self, metabolites, genes, perturbation):
        batch_size = metabolites.shape[0]
        device = metabolites.device
        n_met = self.config['n_metabolites']
        
        state = torch.cat([metabolites, genes], dim=-1)
        full_input = torch.cat([metabolites, genes, perturbation], dim=-1)
        
        # === STOICHIOMETRIC DYNAMICS (NEW IN V23) ===
        dM_stoich, fluxes = self.compute_stoichiometric_dynamics(metabolites, genes, perturbation)
        
        # === PERCEPTION ===
        per_element = full_input.unsqueeze(-1)
        latents = self.perceiver(per_element)
        ssm_out = self.ssm(latents)
        moe_out, aux_loss = self.moe(ssm_out)
        perception = moe_out.mean(dim=1)
        
        # === REASONING ===
        ns_out, predicates, rule_activations = self.neurosymbolic(perception)
        prog_out, satisfaction, weights = self.program_synth(state)
        actions, plans = self.planner(perception, perception)
        
        # === DYNAMICS ===
        koopman_pred = self.koopman(state)
        
        # === COMBINE ===
        combined = torch.cat([perception, ns_out], dim=-1)
        neural_output = self.output_head(combined)
        
        # Blend stoichiometric + neural predictions
        # Stoichiometric provides physical constraints
        # Neural provides learned corrections
        output = 0.5 * neural_output + 0.3 * koopman_pred + 0.2 * prog_out
        
        # Apply stoichiometric dynamics to metabolites
        new_metabolites = metabolites + 0.1 * dM_stoich + 0.05 * output[:, :n_met]
        new_metabolites = torch.clamp(new_metabolites, 0.01, 20.0)
        
        new_genes = genes + 0.05 * output[:, n_met:]
        new_genes = torch.clamp(new_genes, 0.01, 10.0)
        
        uncertainty = self.uncertainty_head(perception)
        
        return {
            'metabolites': new_metabolites,
            'genes': new_genes,
            'uncertainty': uncertainty,
            'fluxes': fluxes,
            'dM_stoich': dM_stoich,
            'predicates': predicates,
            'aux_loss': aux_loss
        }

print("\n" + "="*70)
print("🧬 DarkManifoldV23 — REAL BIOCHEMISTRY + COMPLETE TRAINING")
print("="*70)

In [ ]:
#@title 1️⃣2️⃣ Data Generator (ADDED - was missing in V22!)

def generate_syn3a_trajectory(n_met, n_gene, seq_len, batch_size, S_real=None, condition='growth'):
    """
    Generate biologically plausible trajectories.
    Uses real stoichiometry to simulate dynamics.
    """
    # Initial state (log-normal distribution like real concentrations)
    met_init = torch.exp(torch.randn(batch_size, n_met) * 0.5)
    gene_init = torch.exp(torch.randn(batch_size, n_gene) * 0.3) + 0.5
    
    # Condition-specific perturbations
    if condition == 'growth':
        pert = torch.zeros(batch_size, PERTURBATION_DIM)
        pert[:, 0] = 1.0  # Growth signal
        growth_rate = 0.02
    elif condition == 'stress':
        pert = torch.zeros(batch_size, PERTURBATION_DIM)
        pert[:, 1] = 1.0  # Stress signal
        growth_rate = -0.01
    elif condition == 'knockout':
        pert = torch.zeros(batch_size, PERTURBATION_DIM)
        pert[:, 2] = 1.0  # KO signal
        # Knock out random genes
        ko_mask = torch.rand(batch_size, n_gene) < 0.1
        gene_init[ko_mask] = 0.01
        growth_rate = 0.005
    else:
        pert = torch.randn(batch_size, PERTURBATION_DIM) * 0.1
        growth_rate = 0.01
    
    # Generate trajectory
    met_traj = [met_init]
    gene_traj = [gene_init]
    
    met = met_init.clone()
    gene = gene_init.clone()
    
    for t in range(seq_len - 1):
        # Simple dynamics with noise
        if S_real is not None:
            # Use real stoichiometry
            S = torch.tensor(S_real[:n_met, :min(S_real.shape[1], n_met)], dtype=torch.float32)
            # Random fluxes modulated by gene expression
            v = torch.randn(batch_size, S.shape[1]) * 0.1
            if S.shape[1] <= n_gene:
                v = v * gene[:, :S.shape[1]]  # Gene modulation
            dM = v @ S.T
        else:
            dM = torch.randn_like(met) * 0.05
        
        # Apply dynamics
        met = met + dM + growth_rate * met + torch.randn_like(met) * 0.02
        met = torch.clamp(met, 0.01, 20.0)
        
        # Gene expression changes slowly
        gene = gene + torch.randn_like(gene) * 0.01 * gene
        gene = torch.clamp(gene, 0.01, 10.0)
        
        met_traj.append(met.clone())
        gene_traj.append(gene.clone())
    
    return {
        'metabolites': torch.stack(met_traj, dim=1),  # (batch, seq_len, n_met)
        'genes': torch.stack(gene_traj, dim=1),       # (batch, seq_len, n_gene)
        'perturbation': pert,
        'condition': condition
    }

print("✅ Data generator defined")

In [ ]:
#@title 1️⃣3️⃣ Initialize V23 with REAL Data

config = {
    'n_metabolites': N_METABOLITES,
    'n_genes': N_GENES,
    'n_reactions': N_REACTIONS,
    'perturbation_dim': PERTURBATION_DIM,
    'd_model': D_MODEL,
    'n_latents': N_LATENTS,
    'ssm_dim': SSM_DIM,
    'n_experts': N_EXPERTS,
    'top_k': TOP_K,
    'koopman_dim': KOOPMAN_DIM,
    'max_rules': MAX_RULES,
    'hierarchy_levels': HIERARCHY_LEVELS
}

# Initialize with REAL biochemistry!
model = DarkManifoldV23(
    config,
    real_S=biochem.S,
    real_G=biochem.G
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"\n🧬 V23 initialized with REAL biochemistry")
print(f"   Parameters: {n_params:,}")
print(f"   Stoichiometry: {model.S.shape} (REAL from SBML)")
print(f"   Gene-Reaction: {model.G.shape} (REAL from GPR)")

In [ ]:
#@title 1️⃣4️⃣ Training Loop (ADDED - was missing in V22!)

# OPTIMIZER (was missing in V22!)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, N_EPOCHS, eta_min=1e-6)

# Loss tracking
losses = []
met_corrs = []
gene_corrs = []

print("🚀 Starting training...")
print(f"   Epochs: {N_EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   LR: {LEARNING_RATE}")

for epoch in tqdm(range(N_EPOCHS)):
    model.train()
    
    # Generate batch with different conditions
    conditions = ['growth', 'stress', 'knockout', 'normal']
    condition = conditions[epoch % len(conditions)]
    
    data = generate_syn3a_trajectory(
        N_METABOLITES, N_GENES, SEQ_LEN, BATCH_SIZE,
        S_real=biochem.S, condition=condition
    )
    
    met_seq = data['metabolites'].to(device)
    gene_seq = data['genes'].to(device)
    pert = data['perturbation'].to(device)
    
    total_loss = 0
    pred_mets = []
    pred_genes = []
    
    met = met_seq[:, 0]
    gene = gene_seq[:, 0]
    
    # Unroll through sequence
    for t in range(SEQ_LEN - 1):
        output = model(met, gene, pert)
        
        met_pred = output['metabolites']
        gene_pred = output['genes']
        
        # Targets
        met_target = met_seq[:, t + 1]
        gene_target = gene_seq[:, t + 1]
        
        # Losses
        met_loss = F.mse_loss(met_pred, met_target)
        gene_loss = F.mse_loss(gene_pred, gene_target)
        aux_loss = output['aux_loss']
        
        step_loss = met_loss + 0.5 * gene_loss + 0.01 * aux_loss
        total_loss += step_loss
        
        pred_mets.append(met_pred.detach())
        pred_genes.append(gene_pred.detach())
        
        # Teacher forcing with noise
        if torch.rand(1) < 0.5:  # 50% teacher forcing
            met = met_target + torch.randn_like(met_target) * 0.01
            gene = gene_target + torch.randn_like(gene_target) * 0.01
        else:
            met = met_pred.detach()
            gene = gene_pred.detach()
    
    # Backprop
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    
    # Track metrics
    losses.append(total_loss.item() / SEQ_LEN)
    
    # Compute correlation
    if epoch % 100 == 0:
        pred_met_stack = torch.stack(pred_mets, dim=1)
        target_met = met_seq[:, 1:]
        
        # Flatten and compute correlation
        pred_flat = pred_met_stack.flatten().cpu().numpy()
        target_flat = target_met.flatten().cpu().numpy()
        corr = np.corrcoef(pred_flat, target_flat)[0, 1]
        met_corrs.append(corr)
        
        print(f"\nEpoch {epoch}: Loss={losses[-1]:.4f}, Met Corr={corr:.4f}, Condition={condition}")

In [ ]:
#@title 1️⃣5️⃣ Evaluation & Visualization

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss curve
ax = axes[0, 0]
ax.plot(losses, 'b-', alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# Correlation curve
ax = axes[0, 1]
if met_corrs:
    ax.plot(range(0, len(met_corrs)*100, 100), met_corrs, 'g-o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Correlation')
ax.set_title('Metabolite Prediction Correlation')
ax.grid(True, alpha=0.3)

# Test on different conditions
model.eval()
with torch.no_grad():
    # Growth condition
    data_growth = generate_syn3a_trajectory(N_METABOLITES, N_GENES, 30, 1, biochem.S, 'growth')
    
    met = data_growth['metabolites'][0, 0:1].to(device)
    gene = data_growth['genes'][0, 0:1].to(device)
    pert = data_growth['perturbation'][0:1].to(device)
    
    pred_traj = [met.cpu().numpy()[0]]
    for t in range(29):
        out = model(met, gene, pert)
        met = out['metabolites']
        gene = out['genes']
        pred_traj.append(met.cpu().numpy()[0])
    
    pred_traj = np.array(pred_traj)
    target_traj = data_growth['metabolites'][0].numpy()

# Plot trajectories for key metabolites
ax = axes[1, 0]
for i in range(min(5, N_METABOLITES)):
    ax.plot(pred_traj[:, i], '--', label=f'Pred {i}', alpha=0.7)
    ax.plot(target_traj[:, i], '-', label=f'Target {i}', alpha=0.5)
ax.set_xlabel('Time step')
ax.set_ylabel('Concentration')
ax.set_title('Metabolite Trajectories (Growth)')
ax.legend(ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)

# Scatter plot: predicted vs actual
ax = axes[1, 1]
ax.scatter(target_traj.flatten(), pred_traj.flatten(), alpha=0.3, s=1)
ax.plot([0, target_traj.max()], [0, target_traj.max()], 'r--')
ax.set_xlabel('Target')
ax.set_ylabel('Predicted')
final_corr = np.corrcoef(target_traj.flatten(), pred_traj.flatten())[0, 1]
ax.set_title(f'Prediction Accuracy (r={final_corr:.3f})')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('v23_results.png', dpi=150)
plt.show()

print(f"\n📊 Final Results:")
print(f"   Final Loss: {losses[-1]:.4f}")
print(f"   Final Correlation: {final_corr:.4f}")

In [ ]:
#@title 1️⃣6️⃣ Save Model

torch.save({
    'model_state_dict': model.state_dict(),
    'config': config,
    'losses': losses,
    'met_corrs': met_corrs,
    'biochem_info': {
        'n_metabolites': biochem.n_metabolites,
        'n_reactions': biochem.n_reactions,
        'n_genes': biochem.n_genes,
        'met_ids': biochem.met_ids[:N_METABOLITES],
        'rxn_ids': biochem.rxn_ids[:N_REACTIONS],
        'gene_ids': biochem.gene_ids[:N_GENES]
    }
}, 'dark_manifold_v23.pt')

print("✅ Model saved to dark_manifold_v23.pt")

# 📌 V23 Summary

## What V23 Fixed

| Issue in V22 | V23 Fix |
|-------------|--------|
| ❌ No optimizer | ✅ AdamW + CosineAnnealing |
| ❌ No data generation | ✅ Trajectory generator with conditions |
| ❌ No training loop | ✅ Full unrolled training |
| ❌ Random stoichiometry | ✅ REAL from SBML (iMB155) |
| ❌ No evaluation | ✅ Correlation tracking + visualization |

## Architecture (from V22)
- **Perception:** SSM (Mamba) + Perceiver IO + Sparse MoE
- **Dynamics:** Koopman Operator + Stoichiometric flux model
- **Reasoning:** NeuroSymbolic + Program Synthesis + Hierarchical Planner

## Key Innovation
**Stoichiometric constraint:** Uses REAL biochemistry matrix S from SBML
```
dM/dt = S @ v  (physically constrained)
```
where v is learned flux vector modulated by gene expression.

## Data Source
- Luthey-Schulten Lab, UIUC
- iMB155 model: 304 metabolites, 244 reactions
- Paper: Cell (2022)